# Week 14 Homework - ARIA v9.5: The Resilience Monitor
## Completed Homework Notebook

**Course:** NTU Remote Sensing and Spatial Information Analysis  
**Instructor:** Prof. Su Wen-Ray  
**Case Study:** Landsat Multi-Decadal Trend Analysis and Resilience - Xiulin / Taroko and Taoyuan Plateau  
**Output folder:** `Week14_outputs/`

---

## Submission Summary

This notebook is the completed Week 14 homework submission. It has been executed end to end and keeps the generated outputs inside the notebook. The workflow follows the 0526 ARIA v9.5 lecture:

1. Landsat L5/L7/L8/L9 band harmonization, scale factor correction, and `QA_PIXEL` cloud/shadow masking.
2. 2000-2026/03 annual NDVI trend analysis for Taroko, with 2024 earthquake and 2009 Morakot markers.
3. Pixel-level `linearFit()` NDVI slope map and greening/browning/stable statistics.
4. Taoyuan MNDWI water-frequency mapping, early vs recent pond-loss detection, and 223 known-pond verification.
5. Post-2024 earthquake recovery ratio and resilience-class statistics.
6. Bonus outputs: NDVI/MNDWI/NBR dashboard, 26-year NDVI GIF, and Landsat x Sentinel-2 cross-sensor comparison.

Key results: Taroko uses 878 Landsat images; NDVI trend is +0.00188 NDVI/year; pixel-level classes are 76.1% greening, 10.3% browning, and 13.6% stable; Taoyuan full AOI shows about 426.0 ha net pond/water loss; MNDWI detects 199 of 223 known ponds (89.2%); damaged Taroko pixels have mean recovery ratio 0.415.

---

## Output Gallery

If code-cell outputs are hidden by the notebook viewer, the core figures are shown here directly from `Week14_outputs/`.

### Contact Sheet

![Week14 contact sheet](Week14_outputs/00_week14_contact_sheet.png)

### Core Tasks

![Task 1 annual NDVI](Week14_outputs/01_taroko_ndvi_26yr_timeseries.png)

![Task 2 NDVI slope map](Week14_outputs/02_taroko_ndvi_slope_map.png)

![Task 3 Taoyuan water frequency](Week14_outputs/03_taoyuan_water_frequency.png)

![Task 3 Taoyuan pond change](Week14_outputs/04_taoyuan_pond_change_map.png)

![Task 4 recovery ratio](Week14_outputs/05_taroko_recovery_ratio_map.png)

### Bonus Outputs

![Bonus 1 multi-index dashboard](Week14_outputs/bonus1_multi_index_dashboard.png)

![Bonus 3 cross-sensor line](Week14_outputs/bonus3_cross_sensor_ndvi_line.png)

![Bonus 3 Landsat S2 scatter](Week14_outputs/bonus3_landsat_s2_scatter.png)

![Bonus 2 NDVI GIF](Week14_outputs/bonus2_taroko_ndvi_26yr_timelapse.gif)


---
## S1 — Environment Setup (環境設定) ✅ COMPLETE

Same GEE setup as W13. Replace `'your-project-id'` with your Cloud Project ID.

In [1]:
# ============================================================
# S1 — Environment Setup — COMPLETE
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import ee, geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import os, platform, json, io, requests
from IPython.display import Image as IPImage, display

# ee.Authenticate()  # Uncomment on first run if this machine has no GEE credentials.
EE_PROJECT = os.environ.get('EE_PROJECT', 'helical-analogy-406703')
ee.Initialize(project=EE_PROJECT)

OUTPUT_DIR = Path('Week14_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# This notebook intentionally starts export tasks once during the completed run.
# Set this to False before rerunning if you do not want duplicate Drive exports.
START_GEE_EXPORTS = True
GEE_EXPORT_FOLDER = 'GEE_Exports'

from matplotlib import font_manager as fm
def setup_chinese_font():
    system = platform.system()
    candidates = {
        'Windows': ['Microsoft JhengHei', 'Microsoft YaHei', 'SimHei'],
        'Darwin':  ['PingFang TC', 'Heiti TC', 'STHeiti'],
        'Linux':   ['Noto Sans CJK TC', 'WenQuanYi Micro Hei',
                    'AR PL UMing TW', 'Noto Sans TC']
    }
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates.get(system, candidates['Linux']):
        if name in available:
            plt.rcParams['font.sans-serif'] = [name] + plt.rcParams['font.sans-serif']
            plt.rcParams['axes.unicode_minus'] = False
            return name
    return None
CHINESE_FONT = setup_chinese_font()

def save_ee_thumbnail(image, filename, params):
    """Save a GEE thumbnail; retry and continue if the thumbnail API is slow."""
    path = OUTPUT_DIR / filename
    url = image.getThumbURL(params)
    last_error = None
    for attempt in range(1, 4):
        try:
            response = requests.get(url, timeout=300)
            response.raise_for_status()
            path.write_bytes(response.content)
            print(f"  Saved thumbnail: {path}")
            display(IPImage(filename=str(path)))
            return path
        except Exception as exc:
            last_error = exc
            print(f"  Thumbnail attempt {attempt}/3 failed for {filename}: {exc}")
    if path.exists():
        print(f"  Using existing thumbnail after retry failure: {path}")
        display(IPImage(filename=str(path)))
        return path
    print(f"  WARNING: thumbnail skipped for {filename}: {last_error}")
    return None

def maybe_start_export(image, description, region, scale=30, crs='EPSG:32651'):
    """Create and optionally start a GEE Drive export task."""
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=GEE_EXPORT_FOLDER,
        region=region,
        scale=scale,
        crs=crs,
        maxPixels=1e10,
    )
    if START_GEE_EXPORTS:
        task.start()
        print(f"  Export started: {description} | task id={task.id}")
        print(f"  Status: {task.status().get('state')}")
    else:
        print(f"  Export prepared but not started: {description}")
    return task

# --- Define AOIs ---
# Taroko Focus (Lab 1, 3, 4 — vegetation trends & resilience)
TAROKO_BBOX = [121.34526379253053, 24.046021742135874,
               121.85149217685861, 24.35767637905926]
aoi = ee.Geometry.Rectangle(TAROKO_BBOX)

# Taoyuan Plateau — FULL RANGE (S6 水頻率圖)
# 桃園台地完整範圍：西至新豐/湖口海岸，東至龍潭丘陵
TAOYUAN_BBOX = [120.94, 24.83, 121.35, 25.08]
aoi_taoyuan = ee.Geometry.Rectangle(TAOYUAN_BBOX)

# Taoyuan URBANIZATION CORRIDOR (S6 消失偵測聚焦區)
# 中壢、新屋、高鐵桃園站沿線
TAOYUAN_URBAN_BBOX = [121.00, 24.88, 121.28, 25.05]
aoi_taoyuan_urban = ee.Geometry.Rectangle(TAOYUAN_URBAN_BBOX)

point = ee.Geometry.Point([121.5, 24.2])
elev = ee.Image('USGS/SRTMGL1_003').sample(point, 30).first().get('elevation').getInfo()
print(f"  GEE project: {EE_PROJECT}")
print(f"  Connectivity OK — Elevation: {elev} m")
print(f"  Output folder: {OUTPUT_DIR.resolve()}")
print(f"  AOI 1 (Lab 1/3/4): Taroko Focus — {TAROKO_BBOX}")
print(f"  AOI 2 (Lab 2, S6): Taoyuan Full  — {TAOYUAN_BBOX}")
print(f"  AOI 3 (Lab 2, S6): Taoyuan Urban — {TAOYUAN_URBAN_BBOX}")

  GEE project: helical-analogy-406703
  Connectivity OK — Elevation: 1031 m
  Output folder: C:\Users\Wade\Desktop\ClassPhD\RSmeasure\0526\Week14_outputs
  AOI 1 (Lab 1/3/4): Taroko Focus — [121.34526379253053, 24.046021742135874, 121.85149217685861, 24.35767637905926]
  AOI 2 (Lab 2, S6): Taoyuan Full  — [120.94, 24.83, 121.35, 25.08]
  AOI 3 (Lab 2, S6): Taoyuan Urban — [121.0, 24.88, 121.28, 25.05]


---
## S2 — Landsat Band Harmonization (Landsat 波段統一) ✏️ EXERCISE

Different Landsat missions use different band numbers for the same spectral region.
This completed submission includes the **band renaming functions** so we can merge all four
missions into a single harmonized collection.

| Spectral Region | L5 TM / L7 ETM+ | L8 OLI / L9 OLI-2 | Unified Name |
|----------------|------------------|-------------------|-------------|
| Blue | SR_B1 | SR_B2 | Blue |
| Green | SR_B2 | SR_B3 | Green |
| Red | SR_B3 | SR_B4 | Red |
| NIR | SR_B4 | SR_B5 | NIR |
| SWIR1 | SR_B5 | SR_B6 | SWIR1 |
| SWIR2 | SR_B7 | SR_B7 | SWIR2 |

> **HINT:** `image.select(['old_name1', 'old_name2', ...], ['new_name1', 'new_name2', ...])`

In [2]:
# ============================================================
# S2 — Landsat Band Harmonization — EXERCISE COMPLETE
# ============================================================

def rename_l57(image):
    """Rename Landsat 5/7 bands to unified names."""
    # L5/L7: SR_B1=Blue, SR_B2=Green, SR_B3=Red, SR_B4=NIR, SR_B5=SWIR1, SR_B7=SWIR2
    return (image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'QA_PIXEL'],
        ['Blue',  'Green', 'Red',   'NIR',   'SWIR1', 'SWIR2', 'QA_PIXEL']
    ).copyProperties(image, ['system:time_start']))

def rename_l89(image):
    """Rename Landsat 8/9 bands to unified names."""
    # L8/L9: SR_B2=Blue, SR_B3=Green, SR_B4=Red, SR_B5=NIR, SR_B6=SWIR1, SR_B7=SWIR2
    return (image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'QA_PIXEL'],
        ['Blue',  'Green', 'Red',   'NIR',   'SWIR1', 'SWIR2', 'QA_PIXEL']
    ).copyProperties(image, ['system:time_start']))

def apply_scale_factors(image):
    """Apply Landsat Collection 2 Level 2 scale factors: DN * 0.0000275 + (-0.2)."""
    optical = (image.select(['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2'])
        .multiply(0.0000275).add(-0.2).clamp(0, 1))
    return image.addBands(optical, overwrite=True).copyProperties(image, ['system:time_start'])

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using QA_PIXEL bitmask."""
    qa = image.select('QA_PIXEL')
    # QA_PIXEL: bit 3 = cloud, bit 4 = cloud shadow. 0 means clear.
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    shadow = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud.And(shadow)).copyProperties(image, ['system:time_start'])

# --- Load and merge (COMPLETE) ---
DATE_START = '2000-01-01'
DATE_END = '2026-03-31'   # W14 uses data available through 2026/03 in this course folder.

l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterDate(DATE_START, '2012-12-31').filterBounds(aoi).map(rename_l57)
l7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterDate(DATE_START, DATE_END).filterBounds(aoi).map(rename_l57)
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2013-01-01', DATE_END).filterBounds(aoi).map(rename_l89)
l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2021-01-01', DATE_END).filterBounds(aoi).map(rename_l89)

landsat_all = l5.merge(l7).merge(l8).merge(l9)
landsat_clean = landsat_all.map(mask_landsat_clouds).map(apply_scale_factors)

mission_counts = {
    'L5': l5.size().getInfo(),
    'L7': l7.size().getInfo(),
    'L8': l8.size().getInfo(),
    'L9': l9.size().getInfo(),
}
total_landsat = landsat_clean.size().getInfo()
print(f"  Total Landsat images (2000–2026/03): {total_landsat}")
print(f"  L5: {mission_counts['L5']} | L7: {mission_counts['L7']} | "
      f"L8: {mission_counts['L8']} | L9: {mission_counts['L9']}")
print("  Harmonized bands:", landsat_clean.first().bandNames().getInfo())

  Total Landsat images (2000–2026/03): 878
  L5: 178 | L7: 387 | L8: 233 | L9: 80


  Harmonized bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'QA_PIXEL']


---
## S3 — NDVI & MNDWI Index Calculation (指標計算) ✏️ EXERCISE

Compute two spectral indices from the harmonized Landsat collection:

| Index | Formula | Detects |
|-------|---------|---------|
| **NDVI** | (NIR − Red) / (NIR + Red) | Vegetation health |
| **MNDWI** | (Green − SWIR1) / (Green + SWIR1) | Water bodies |

> **HINT:** Use `normalizedDifference(['band1', 'band2'])` — it computes (b1−b2)/(b1+b2)

In [3]:
# ============================================================
# S3 — Compute NDVI, MNDWI, and NBR — EXERCISE COMPLETE
# ============================================================

def add_indices(image):
    """Add vegetation, water, and disturbance-related spectral indices."""
    # NDVI = (NIR - Red) / (NIR + Red), vegetation greenness.
    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')

    # MNDWI = (Green - SWIR1) / (Green + SWIR1), water detection with built-up suppression.
    mndwi = image.normalizedDifference(['Green', 'SWIR1']).rename('MNDWI')

    # NBR = (NIR - SWIR2) / (NIR + SWIR2), useful for disturbance/bare-surface signals.
    nbr = image.normalizedDifference(['NIR', 'SWIR2']).rename('NBR')

    return image.addBands([ndvi, mndwi, nbr]).copyProperties(image, ['system:time_start'])

landsat_idx = landsat_clean.map(add_indices)
print(f"  Collection with indices: {landsat_idx.size().getInfo()} images")
print("  Added bands: NDVI (vegetation) + MNDWI (water) + NBR (disturbance)")

  Collection with indices: 878 images
  Added bands: NDVI (vegetation) + MNDWI (water) + NBR (disturbance)


---
## S4 — Annual NDVI Time Series (26-Year) ✏️ EXERCISE

Compute **annual median NDVI** for each year from 2000 to 2026.
The function is provided — your job is to create the plot with:
1. Mean, min, max lines (like W13 S4)
2. A linear trend line
3. Event markers for the 2024 earthquake and 2009 Typhoon Morakot

In [4]:
# ============================================================
# S4 — Annual NDVI Composites — COMPLETE
# ============================================================

def compute_annual_ndvi(collection, aoi, start_year=2000, end_year=2026):
    """Return annual median NDVI mean/min/max and valid image count."""
    results = []
    for year in range(start_year, end_year + 1):
        annual = collection.filterDate(f'{year}-01-01', f'{year + 1}-01-01').select('NDVI')
        n = annual.size().getInfo()
        if n == 0:
            results.append((year, None, None, None, n))
            print(f'  {year}: no data (n={n})')
            continue
        median_img = annual.median()
        stats = median_img.reduceRegion(
            reducer=ee.Reducer.mean()
                .combine(ee.Reducer.min(), sharedInputs=True)
                .combine(ee.Reducer.max(), sharedInputs=True),
            geometry=aoi, scale=100, maxPixels=1e9, tileScale=4
        ).getInfo()
        v_mean = stats.get('NDVI_mean')
        v_min  = stats.get('NDVI_min')
        v_max  = stats.get('NDVI_max')
        results.append((year, v_mean, v_min, v_max, n))
        if v_mean is not None:
            print(f'  {year}: mean={v_mean:.4f}  min={v_min:.4f}  max={v_max:.4f}  (n={n})')
        else:
            print(f'  {year}: no valid pixels after masking (n={n})')
    return results

print('Computing 26-year annual NDVI...')
annual_data = compute_annual_ndvi(landsat_idx, aoi)

annual_ndvi_df = pd.DataFrame(annual_data, columns=['year', 'mean_ndvi', 'min_ndvi', 'max_ndvi', 'image_count'])
annual_ndvi_df.to_csv(OUTPUT_DIR / 'task1_annual_ndvi_stats.csv', index=False, encoding='utf-8-sig')

valid_df = annual_ndvi_df.dropna(subset=['mean_ndvi']).copy()
years  = valid_df['year'].tolist()
means  = valid_df['mean_ndvi'].tolist()
mins   = valid_df['min_ndvi'].tolist()
maxs   = valid_df['max_ndvi'].tolist()

print(f"\n  Saved annual NDVI table: {OUTPUT_DIR / 'task1_annual_ndvi_stats.csv'}")
print(f"  Lowest mean NDVI year: {int(valid_df.loc[valid_df['mean_ndvi'].idxmin(), 'year'])}")
print(f"  Highest mean NDVI year: {int(valid_df.loc[valid_df['mean_ndvi'].idxmax(), 'year'])}")

Computing 26-year annual NDVI...


  2000: mean=0.5256  min=-0.2411  max=0.9988  (n=31)


  2001: mean=0.5192  min=-0.3039  max=0.8965  (n=33)


  2002: mean=0.5148  min=-0.2356  max=1.0000  (n=31)


  2003: mean=0.5201  min=-0.2806  max=0.8905  (n=28)


...

  2022: mean=0.5886  min=-0.5154  max=1.0000  (n=54)


  2023: mean=0.5662  min=-0.4194  max=1.0000  (n=59)


  2024: mean=0.5607  min=-1.0000  max=1.0000  (n=38)


  2025: mean=0.5427  min=-1.0000  max=1.0000  (n=33)


  2026: mean=0.4728  min=-1.0000  max=1.0000  (n=8)

  Saved annual NDVI table: Week14_outputs\task1_annual_ndvi_stats.csv
  Lowest mean NDVI year: 2026
  Highest mean NDVI year: 2022


In [5]:
# ============================================================
# S4 (continued) — Plot 26-Year NDVI — EXERCISE COMPLETE
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Spatial spread of annual median NDVI across the AOI.
ax.fill_between(years, mins, maxs, alpha=0.15, color='green',
                label='Spatial range (min–max)')

# Plot annual mean NDVI.
ax.plot(years, means, 'o-', color='green', markersize=6, linewidth=2,
        label='Annual median NDVI', zorder=3)

# Linear trend line across all available years.
z = np.polyfit(years, means, 1)
p = np.poly1d(z)
ax.plot(years, p(years), ':', color='navy', linewidth=2,
        label=f'Trend: {z[0]:+.5f}/yr')

# Major event markers from the lecture.
ax.axvline(2024, color='red', linestyle='--', linewidth=2, label='2024 Hualien EQ')
ax.axvline(2009, color='orange', linestyle='--', linewidth=1.5, label='2009 Morakot')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('NDVI', fontsize=12)
ax.set_title('Taroko 26-Year NDVI Time Series (2000–2026/03)\n'
             '太魯閣 26 年 NDVI 時間序列', fontsize=14)
ax.set_xticks(years[::2])
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()

task1_fig = OUTPUT_DIR / '01_taroko_ndvi_26yr_timeseries.png'
plt.savefig(task1_fig, dpi=180, bbox_inches='tight')
plt.show()

ndvi_trend_slope = float(z[0])
ndvi_total_change = float(z[0] * (max(years) - min(years)))
print(f'\n  26-year trend: {ndvi_trend_slope:+.5f} NDVI/year')
print(f'  Trend-estimated change: {ndvi_total_change:+.4f} from {min(years)} to {max(years)}')
print(f'  Saved figure: {task1_fig}')

<Figure size 1400x600 with 1 Axes>


  26-year trend: +0.00188 NDVI/year
  Trend-estimated change: +0.0488 from 2000 to 2026
  Saved figure: Week14_outputs\01_taroko_ndvi_26yr_timeseries.png


In [6]:
# ============================================================
# S4b — Multi-Index Dashboard (NDVI + MNDWI + NBR) — BONUS
# ============================================================

def annual_multi_index_stats(collection, aoi, start_year=2000, end_year=2026):
    rows = []
    for year in range(start_year, end_year + 1):
        annual = collection.filterDate(f'{year}-01-01', f'{year + 1}-01-01').select(['NDVI', 'MNDWI', 'NBR'])
        n = annual.size().getInfo()
        if n == 0:
            rows.append({'year': year, 'NDVI': None, 'MNDWI': None, 'NBR': None, 'image_count': 0})
            continue
        stats = annual.median().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi, scale=100, maxPixels=1e9, tileScale=4
        ).getInfo()
        rows.append({
            'year': year,
            'NDVI': stats.get('NDVI'),
            'MNDWI': stats.get('MNDWI'),
            'NBR': stats.get('NBR'),
            'image_count': n,
        })
    return pd.DataFrame(rows)

multi_index_df = annual_multi_index_stats(landsat_idx, aoi)
multi_index_df.to_csv(OUTPUT_DIR / 'bonus1_multi_index_timeseries.csv',
                      index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, idx, color in zip(axes, ['NDVI', 'MNDWI', 'NBR'], ['forestgreen', 'steelblue', 'darkorange']):
    ax.plot(multi_index_df['year'], multi_index_df[idx], 'o-', color=color, linewidth=1.8, markersize=4)
    idx_vals = multi_index_df[['year', idx]].dropna()
    if len(idx_vals) >= 2:
        zz = np.polyfit(idx_vals['year'], idx_vals[idx], 1)
        pp = np.poly1d(zz)
        ax.plot(idx_vals['year'], pp(idx_vals['year']), ':', color='black', alpha=0.65,
                label=f'{idx} trend {zz[0]:+.5f}/yr')
    ax.axvline(2024, color='red', linestyle='--', linewidth=1.5, alpha=0.75)
    ax.set_ylabel(idx)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)
axes[-1].set_xlabel('Year')
fig.suptitle('Taroko Multi-Index Dashboard (2000–2026/03)\nNDVI / MNDWI / NBR 年度平均指標', fontsize=14)
plt.tight_layout()

dashboard_fig = OUTPUT_DIR / 'bonus1_multi_index_dashboard.png'
plt.savefig(dashboard_fig, dpi=180, bbox_inches='tight')
plt.show()
print(f"  Saved dashboard: {dashboard_fig}")

<Figure size 1300x900 with 3 Axes>

  Saved dashboard: Week14_outputs\bonus1_multi_index_dashboard.png


### Task 1 Analysis

## Task 1: Landsat Harmonization And 26-Year NDVI

![26-year NDVI](Week14_outputs/01_taroko_ndvi_26yr_timeseries.png)

Landsat 四代影像完成波段調和後，所有影像皆具有 `Blue, Green, Red, NIR, SWIR1, SWIR2, QA_PIXEL` 共同 band name，因此 NDVI 可用一致公式 `(NIR - Red) / (NIR + Red)` 計算。年度 NDVI 以每年 median composite 後取 AOI 空間平均。

主要結果：

| 指標 | 數值 |
|---|---:|
| Landsat total images | 878 |
| L5 / L7 / L8 / L9 | 178 / 387 / 233 / 80 |
| 26-year NDVI trend | +0.00188 NDVI/year |
| Trend-estimated total change | +0.0488 |
| Highest annual mean NDVI | 2022, 0.5886 |
| Lowest annual mean NDVI | 2026, 0.4728 |
| 2024 earthquake year mean NDVI | 0.5607 |

2000-2026/03 的整體趨勢是緩慢 greening。2005、2007、2012 與 2026 有較低 NDVI，其中 2026 只到 3 月，影像數僅 8 張，受季節與資料不完整影響，不應直接視為全年退化。2009 莫拉克年平均 NDVI 為 0.5152，2024 地震年為 0.5607；兩者在 AOI 年平均折線上並非最強低谷，說明災害訊號被大面積森林與季節差異稀釋。因此，事件影響應搭配逐像素趨勢、ΔNDVI 與 recovery ratio 判讀，而不是只看整體平均。

---
## S5 — Pixel-Level Trend Map (逐像素趨勢圖) ✏️ EXERCISE

Apply `linearFit` to every pixel to map **where** vegetation is greening vs browning.

> **HINT:** This is the same `linearFit` from W13 D8, but applied to 26 years of Landsat.

In [7]:
# ============================================================
# S5 — Pixel-Level Trend — EXERCISE COMPLETE
# ============================================================

def annual_ndvi_image(year):
    """Create annual median NDVI image with a year band for linear regression."""
    year = ee.Number(year).int()
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')
    median_ndvi = landsat_idx.filterDate(start, end).select('NDVI').median()
    time_band = ee.Image.constant(year).float().rename('time')
    return median_ndvi.addBands(time_band).set('system:time_start', start.millis())

year_list = ee.List.sequence(2000, 2026)
annual_col = ee.ImageCollection(year_list.map(annual_ndvi_image))

# linearFit: dependent = NDVI, independent = time.
trend = annual_col.select(['time', 'NDVI']).reduce(ee.Reducer.linearFit())
slope = trend.select('scale').rename('ndvi_slope_per_year')

# --- Trend class statistics ---
greening = slope.gt(0.001)
browning = slope.lt(-0.001)
stable = slope.gte(-0.001).And(slope.lte(0.001))
pixel_area = ee.Image.pixelArea()

def area_km2(mask):
    value = pixel_area.updateMask(mask).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi, scale=30, maxPixels=1e10, tileScale=4
    ).get('area')
    return ee.Number(value).divide(1e6)

valid_area_km2 = area_km2(slope.mask())
trend_stats = {
    'greening_km2': area_km2(greening).getInfo(),
    'browning_km2': area_km2(browning).getInfo(),
    'stable_km2': area_km2(stable).getInfo(),
    'valid_km2': valid_area_km2.getInfo(),
}
for key in ['greening', 'browning', 'stable']:
    trend_stats[f'{key}_pct'] = trend_stats[f'{key}_km2'] / trend_stats['valid_km2'] * 100
pd.DataFrame([trend_stats]).to_csv(OUTPUT_DIR / 'task2_trend_area_stats.csv',
                                  index=False, encoding='utf-8-sig')

print("  Pixel-level NDVI trend statistics (threshold ±0.001 NDVI/year):")
for key in ['greening', 'browning', 'stable']:
    print(f"  {key:9s}: {trend_stats[f'{key}_km2']:.2f} km² ({trend_stats[f'{key}_pct']:.1f}%)")

# --- Visualization ---
vis_trend = {
    'min': -0.01, 'max': 0.01,
    'palette': ['d73027', 'fc8d59', 'fee08b', 'ffffbf',
                'd9ef8b', '91cf60', '1a9850']
}

save_ee_thumbnail(
    slope.clip(aoi),
    '02_taroko_ndvi_slope_map.png',
    {
        'region': aoi,
        'dimensions': 900,
        'min': vis_trend['min'],
        'max': vis_trend['max'],
        'palette': vis_trend['palette'],
    },
)

Map5 = geemap.Map(center=[24.20, 121.60], zoom=10)
Map5.addLayer(slope.clip(aoi), vis_trend, 'NDVI Trend (slope/year)')
Map5.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map5

# GeoTIFF export required by the homework.
task_trend = maybe_start_export(
    image=trend.clip(aoi).toFloat(),
    description='Week14_taroko_ndvi_trend_26yr',
    region=aoi,
    scale=30,
    crs='EPSG:32651',
)

  Pixel-level NDVI trend statistics (threshold ±0.001 NDVI/year):
  greening : 1351.34 km² (76.1%)
  browning : 183.11 km² (10.3%)
  stable   : 240.84 km² (13.6%)


  Saved thumbnail: Week14_outputs\02_taroko_ndvi_slope_map.png


<IPython.core.display.Image object>

  Export started: Week14_taroko_ndvi_trend_26yr | task id=5PYRDYRWACXGHZLZHTZMO3AF


  Status: READY


---
## S6 — Taoyuan Pond Disappearance (桃園埤塘消失偵測) ✏️ EXERCISE

桃園台地在日治時期桃園大圳興建前，曾擁有約 **6,000–8,000 口埤塘**（農田水利署），是台灣規模最大的灌溉水塘景觀。
隨著都市化發展，大量埤塘被填平，目前僅存約 3,000 口。

Compare water presence between **early** (2000–2005) and **recent** (2021–2026)
periods to detect which ponds have **survived**, **disappeared**, or **appeared**.

MNDWI > 0.1 → classified as water

> **NOTE:** We need a separate Landsat collection for Taoyuan (different AOI from Taroko).
> **HINT:** Compute median MNDWI for each period, then threshold at 0.1.

In [8]:
# ============================================================
# S6 — Taoyuan Pond Disappearance — EXERCISE COMPLETE
#       + True-Color Verification Layers
# ============================================================

# --- Load Landsat for Taoyuan (same functions from S2) ---
l5_ty = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterDate(DATE_START, '2012-12-31').filterBounds(aoi_taoyuan).map(rename_l57)
l7_ty = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterDate(DATE_START, DATE_END).filterBounds(aoi_taoyuan).map(rename_l57)
l8_ty = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2013-01-01', DATE_END).filterBounds(aoi_taoyuan).map(rename_l89)
l9_ty = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2021-01-01', DATE_END).filterBounds(aoi_taoyuan).map(rename_l89)

landsat_taoyuan = (l5_ty.merge(l7_ty).merge(l8_ty).merge(l9_ty)
                   .map(mask_landsat_clouds)
                   .map(apply_scale_factors)
                   .map(add_indices))

print(f'  Taoyuan Landsat images: {landsat_taoyuan.size().getInfo()}')

# Annual water frequency: MNDWI > 0.1 for each year.
def yearly_water(year):
    year = ee.Number(year).int()
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')
    annual_mndwi = landsat_taoyuan.filterDate(start, end).select('MNDWI').median()
    return annual_mndwi.gt(0.1).rename('water').set('system:time_start', start.millis())

water_years = ee.List.sequence(2000, 2026)
water_col = ee.ImageCollection(water_years.map(yearly_water))
water_frequency = water_col.mean().rename('water_frequency')

# Early/recent water masks.
early_water = (landsat_taoyuan
    .filterDate('2000-01-01', '2005-12-31')
    .select('MNDWI').median()
    .gt(0.1).rename('early_water'))

recent_water = (landsat_taoyuan
    .filterDate('2021-01-01', DATE_END)
    .select('MNDWI').median()
    .gt(0.1).rename('recent_water'))

# Detect changes.
water_loss = early_water.And(recent_water.Not()).selfMask()
water_gain = recent_water.And(early_water.Not()).selfMask()
stable_water = early_water.And(recent_water).selfMask()

def water_area_stats(region, label):
    pa = ee.Image.pixelArea()
    def ha(mask):
        return ee.Number(pa.updateMask(mask).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=region, scale=30, maxPixels=1e10, tileScale=4
        ).get('area')).divide(10000)
    row = {
        'region': label,
        'stable_water_ha': ha(stable_water).getInfo(),
        'lost_ponds_ha': ha(water_loss).getInfo(),
        'new_water_ha': ha(water_gain).getInfo(),
    }
    row['net_change_ha'] = row['new_water_ha'] - row['lost_ponds_ha']
    return row

taoyuan_full_stats = water_area_stats(aoi_taoyuan, 'Taoyuan full')
taoyuan_urban_stats = water_area_stats(aoi_taoyuan_urban, 'Taoyuan urban corridor')
pond_area_df = pd.DataFrame([taoyuan_full_stats, taoyuan_urban_stats])
pond_area_df.to_csv(OUTPUT_DIR / 'task3_pond_area_stats.csv', index=False, encoding='utf-8-sig')
print("  Pond area change stats (ha):")
display(pond_area_df)

# Download and use 223 known pond points for verification.
pond_geojson_path = OUTPUT_DIR / 'taoyuan_ponds_223.geojson'
if not pond_geojson_path.exists():
    url = 'https://drive.google.com/uc?export=download&id=1qwrIIELIJXbrBL_oCBTcoE-aoWq1bdXw'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    pond_geojson_path.write_bytes(r.content)

with open(pond_geojson_path, encoding='utf-8') as f:
    ponds_geojson = json.load(f)

pond_features = []
for feat in ponds_geojson['features']:
    geom = ee.Geometry(feat['geometry'])
    if feat['geometry']['type'] != 'Point':
        geom = geom.centroid(1)
    pond_features.append(ee.Feature(geom, feat.get('properties', {})))

ponds_fc = ee.FeatureCollection(pond_features)
mndwi_at_ponds = recent_water.unmask(0).sampleRegions(collection=ponds_fc, scale=30, geometries=True)
detected = mndwi_at_ponds.filter(ee.Filter.eq('recent_water', 1)).size().getInfo()
total_ponds = len(pond_features)
pond_detection_rate = detected / total_ponds * 100
print(f"  Known ponds: {total_ponds}")
print(f"  MNDWI detected in recent period: {detected}")
print(f"  Detection rate: {pond_detection_rate:.1f}%")

pd.DataFrame([{
    'known_ponds': total_ponds,
    'mndwi_detected': detected,
    'detection_rate_pct': pond_detection_rate,
}]).to_csv(OUTPUT_DIR / 'task3_pond_detection_rate.csv', index=False, encoding='utf-8-sig')

# ── True-Color Verification Layers ──────────────────────────
def mask_s2_clouds_ty(image):
    scl = image.select('SCL')
    return image.updateMask(scl.gte(4).And(scl.lte(7)))

s2_taoyuan = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_taoyuan_urban)
    .filterDate('2024-01-01', '2025-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds_ty))

s2_recent_rgb = s2_taoyuan.select(['B4', 'B3', 'B2']).median()

early_rgb = (landsat_taoyuan
    .filterDate('2000-01-01', '2005-12-31')
    .select(['Red', 'Green', 'Blue']).median())

change_class = (ee.Image(0)
    .where(stable_water, 1)
    .where(water_loss, 2)
    .where(water_gain, 3)
    .selfMask()
    .rename('pond_change_class'))

save_ee_thumbnail(
    water_frequency.clip(aoi_taoyuan),
    '03_taoyuan_water_frequency.png',
    {
        'region': aoi_taoyuan,
        'dimensions': 900,
        'min': 0,
        'max': 1,
        'palette': ['f7fbff', 'deebf7', '9ecae1', '3182bd', '08519c'],
    },
)
save_ee_thumbnail(
    change_class.clip(aoi_taoyuan_urban),
    '04_taoyuan_pond_change_map.png',
    {
        'region': aoi_taoyuan_urban,
        'dimensions': 900,
        'min': 1,
        'max': 3,
        'palette': ['0000FF', 'FF0000', '00FF00'],
    },
)

# Visualization — focused on 中壢/新屋/高鐵 urbanization corridor
Map6 = geemap.Map(center=[24.96, 121.14], zoom=12)
Map6.addLayer(water_frequency.clip(aoi_taoyuan), {
    'min': 0, 'max': 1,
    'palette': ['f7fbff', 'deebf7', '9ecae1', '3182bd', '08519c']
}, 'Water frequency 水頻率 2000–2026')
Map6.addLayer(early_rgb.clip(aoi_taoyuan_urban),
              {'bands': ['Red', 'Green', 'Blue'], 'min': 0, 'max': 0.25},
              'Landsat 早期真彩色 2000–2005 (30m)')
Map6.addLayer(s2_recent_rgb.clip(aoi_taoyuan_urban),
              {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500},
              'S2 近期真彩色 2024–2025 (10m)')
Map6.addLayer(stable_water.clip(aoi_taoyuan_urban), {'palette': ['0000FF']},
              'Stable ponds 存活埤塘')
Map6.addLayer(water_gain.clip(aoi_taoyuan_urban), {'palette': ['00FF00']},
              'New water 新增水體')
Map6.addLayer(water_loss.clip(aoi_taoyuan_urban), {'palette': ['FF0000']},
              'Lost ponds 消失埤塘')
Map6.addLayer(aoi_taoyuan_urban, {'color': 'orange'}, 'AOI — 中壢/新屋/高鐵走廊')
Map6.addLayer(aoi_taoyuan, {'color': 'yellow'}, 'AOI — Taoyuan Full')
Map6

  Taoyuan Landsat images: 2485


  Pond area change stats (ha):


,region,stable_water_ha,lost_ponds_ha,new_water_ha,net_change_ha
0,Taoyuan full,19047.558268,588.796728,162.802015,-425.994713
1,Taoyuan urban corridor,4034.774858,352.194573,50.627679,-301.566894


  Known ponds: 223
  MNDWI detected in recent period: 199
  Detection rate: 89.2%


  Saved thumbnail: Week14_outputs\03_taoyuan_water_frequency.png


<IPython.core.display.Image object>

  Saved thumbnail: Week14_outputs\04_taoyuan_pond_change_map.png


<IPython.core.display.Image object>

Map(center=[24.96, 121.14], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

### Task 3 Analysis

## Task 3: Taoyuan Pond Disappearance With MNDWI

![Taoyuan water frequency](Week14_outputs/03_taoyuan_water_frequency.png)

![Taoyuan pond change](Week14_outputs/04_taoyuan_pond_change_map.png)

桃園分析使用 MNDWI `(Green - SWIR1) / (Green + SWIR1)`，並以 `MNDWI > 0.1` 判為水體。先建立 2000-2026/03 每年水體 mask，再計算水頻率圖；接著比較 early period（2000-2005）與 recent period（2021-2026/03）之 median MNDWI water mask。

面積結果：

| Region | Stable water (ha) | Lost ponds (ha) | New water (ha) | Net change (ha) |
|---|---:|---:|---:|---:|
| Taoyuan full | 19047.56 | 588.80 | 162.80 | -425.99 |
| Urban corridor | 4034.77 | 352.19 | 50.63 | -301.57 |

完整桃園 AOI 的淨水面變化為 **-426.0 ha**，其中中壢/新屋/高鐵走廊就佔 **-301.6 ha**，顯示消失埤塘高度集中在都市化走廊。新增水體面積小於消失面積，可能包含新設滯洪池、工程開挖積水、整修後農塘或季節性水田；因此不宜直接把所有 green 像元解讀成永久水塘增加。

驗證結果：

| Known ponds | MNDWI detected | Detection rate |
|---:|---:|---:|
| 223 | 199 | 89.2% |

MNDWI 對大部分已知埤塘有效，但可能漏掉小面積水體、混合像元、植被覆蓋水面或陰影干擾。都市防洪上，埤塘被填平代表分散式蓄洪容量下降，不透水面增加，使短延時強降雨下的逕流尖峰更高，排水系統更容易超載。這也是講義強調桃園埤塘不只是文化景觀，也是都市韌性基礎設施的原因。

---
## S7 — Vegetation Resilience (植被韌性) ✏️ EXERCISE

Compute the **recovery ratio** to measure how well vegetation is recovering
after the 2024 earthquake.

| Phase | Period | Purpose |
|-------|--------|---------|
| Baseline | 2020–2024/03 | Pre-disturbance reference |
| Impact | 2024/04–2024/12 | Immediate damage |
| Recovery | 2025/06–2026/03 | Current state |

**Recovery ratio = (Recovery − Impact) / (Baseline − Impact)**

In [9]:
# ============================================================
# S7 — Resilience Metrics — EXERCISE COMPLETE
# ============================================================

# Periods follow the W14 lecture: baseline before 2024/04/03, impact after EQ,
# and recovery after one growing season.
baseline = landsat_idx.filterDate('2020-01-01', '2024-03-31').select('NDVI').median()
impact   = landsat_idx.filterDate('2024-04-01', '2024-12-31').select('NDVI').median()
recovery = landsat_idx.filterDate('2025-06-01', DATE_END).select('NDVI').median()

# Recovery ratio = (Recovery - Impact) / (Baseline - Impact)
impact_mag = baseline.subtract(impact).rename('impact_magnitude')
recovery_mag = recovery.subtract(impact).rename('recovery_magnitude')

# Only evaluate pixels with meaningful NDVI damage (> 0.1).
damage_mask = impact_mag.gt(0.1)
recovery_ratio = (recovery_mag.divide(impact_mag)
                  .rename('recovery_ratio')
                  .clamp(-1, 2)
                  .updateMask(damage_mask))

# Recovery classes.
degrading = recovery_ratio.lt(0)
slow = recovery_ratio.gte(0).And(recovery_ratio.lt(0.5))
recovering = recovery_ratio.gte(0.5).And(recovery_ratio.lte(1.0))
exceeded = recovery_ratio.gt(1.0)

def rr_area_km2(mask):
    return ee.Number(ee.Image.pixelArea().updateMask(mask).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=30,
        maxPixels=1e10, tileScale=4
    ).get('area')).divide(1e6)

damaged_km2 = rr_area_km2(damage_mask).getInfo()
rr_stats = {
    'damaged_km2': damaged_km2,
    'degrading_km2': rr_area_km2(degrading).getInfo(),
    'slow_km2': rr_area_km2(slow).getInfo(),
    'recovering_km2': rr_area_km2(recovering).getInfo(),
    'exceeded_km2': rr_area_km2(exceeded).getInfo(),
}
for key in ['degrading', 'slow', 'recovering', 'exceeded']:
    rr_stats[f'{key}_pct'] = rr_stats[f'{key}_km2'] / damaged_km2 * 100 if damaged_km2 else np.nan

mean_rr = recovery_ratio.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi, scale=30, maxPixels=1e10, tileScale=4
).get('recovery_ratio').getInfo()
rr_stats['mean_recovery_ratio'] = mean_rr
pd.DataFrame([rr_stats]).to_csv(OUTPUT_DIR / 'task4_recovery_ratio_stats.csv',
                               index=False, encoding='utf-8-sig')

print(f"  Damaged pixels evaluated: {damaged_km2:.2f} km²")
print(f"  Mean recovery ratio: {mean_rr:.3f}")
for key, label in [('degrading', 'degrading (<0)'), ('slow', 'slow (0–0.5)'),
                   ('recovering', 'recovering (0.5–1.0)'), ('exceeded', 'exceeded (>1.0)')]:
    print(f"  {label:20s}: {rr_stats[f'{key}_km2']:.2f} km² ({rr_stats[f'{key}_pct']:.1f}%)")

# Visualization
vis_rec = {'min': -0.2, 'max': 1.4,
           'palette': ['8B0000', 'FF4500', 'FFD700', '90EE90', '228B22', '1E90FF']}

save_ee_thumbnail(
    recovery_ratio.clip(aoi),
    '05_taroko_recovery_ratio_map.png',
    {
        'region': aoi,
        'dimensions': 900,
        'min': vis_rec['min'],
        'max': vis_rec['max'],
        'palette': vis_rec['palette'],
    },
)

Map7 = geemap.Map(center=[24.20, 121.60], zoom=11)
Map7.addLayer(recovery_ratio.clip(aoi), vis_rec, 'Recovery Ratio')
Map7.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map7

task_recovery = maybe_start_export(
    image=recovery_ratio.clip(aoi).toFloat(),
    description='Week14_taroko_recovery_ratio',
    region=aoi,
    scale=30,
    crs='EPSG:32651',
)

  Damaged pixels evaluated: 91.94 km²
  Mean recovery ratio: 0.415
  degrading (<0)      : 16.15 km² (17.6%)
  slow (0–0.5)        : 36.36 km² (39.5%)
  recovering (0.5–1.0): 27.17 km² (29.6%)
  exceeded (>1.0)     : 11.93 km² (13.0%)


  Saved thumbnail: Week14_outputs\05_taroko_recovery_ratio_map.png


<IPython.core.display.Image object>

  Export started: Week14_taroko_recovery_ratio | task id=ZQYPFAIMYPZW6CCIQP3HGBGC


  Status: READY


### Task 4 Analysis

## Task 4: Vegetation Resilience Metrics

![Recovery ratio](Week14_outputs/05_taroko_recovery_ratio_map.png)

依講義定義三期：

- **Baseline：** 2020-01-01 至 2024-03-31
- **Impact：** 2024-04-01 至 2024-12-31
- **Recovery：** 2025-06-01 至 2026-03-31

公式：

```text
Recovery Ratio = (Recovery_NDVI - Impact_NDVI) / (Baseline_NDVI - Impact_NDVI)
```

本作業只評估 `Baseline - Impact > 0.1` 的顯著受損像元。結果如下：

| Class | Area (km²) | Percent |
|---|---:|---:|
| Degrading (< 0) | 16.15 | 17.6% |
| Slow recovery (0-0.5) | 36.36 | 39.5% |
| Recovering (0.5-1.0) | 27.17 | 29.6% |
| Exceeded (> 1.0) | 11.93 | 13.0% |
| Damaged pixels evaluated | 91.94 | 100.0% |
| Mean recovery ratio | 0.415 | |

受損像元中，退化與慢速恢復合計 **57.1%**，代表超過一半的顯著受損地區尚未恢復到震前一半以上。這些區域多半是陡峭峽谷壁、河道兩側裸露堆積、道路切坡、崩塌源頭或持續受沖刷的坡面；即使經過 2025-2026 的恢復期，NDVI 仍偏低。相對地，recovering 與 exceeded 區域可能位於較緩坡面、谷地邊緣或仍保有土壤的崩塌堆積面，植被可快速長草或灌木化。

復育建議是分級處理：靠近道路、溪流、聚落或保全對象，且 recovery ratio 低的地區應優先工程穩定與主動復育；recovery ratio 已高且遠離保全對象的區域，則可採自然恢復監測，避免過度介入。

GeoTIFF Drive export 已啟動並完成：

- `Week14_taroko_recovery_ratio` - `SUCCEEDED`

---
## S8a — Bonus: 26-Year NDVI Time-Lapse GIF（26 年 NDVI 動畫）

除了靜態年度圖，本段用 GEE thumbnail API 產生 2000–2026/03 的年度 NDVI GIF。
動畫可快速檢查是否有缺年、異常雲遮罩，以及 2009、2024 等事件年是否在視覺上形成明顯低 NDVI 訊號。

In [10]:
# ============================================================
# S8a — 26-Year NDVI Time-Lapse GIF — BONUS
# ============================================================
from PIL import Image, ImageDraw, ImageFont
import imageio.v2 as imageio

gif_frames = []
frame_dir = OUTPUT_DIR / 'bonus2_ndvi_frames'
frame_dir.mkdir(exist_ok=True)

ndvi_palette = ['8c510a', 'd8b365', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e']
for year in range(2000, 2027):
    composite = landsat_idx.filterDate(f'{year}-01-01', f'{year + 1}-01-01').select('NDVI').median()
    params = {
        'region': aoi,
        'dimensions': 512,
        'min': 0,
        'max': 0.8,
        'palette': ndvi_palette,
    }
    url = composite.clip(aoi).getThumbURL(params)
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    img = Image.open(io.BytesIO(response.content)).convert('RGB')

    draw = ImageDraw.Draw(img)
    label = f'{year}'
    if year == 2009:
        label += '  Morakot'
    if year == 2024:
        label += '  Hualien EQ'
    draw.rectangle((8, 8, 238, 38), fill=(0, 0, 0))
    draw.text((14, 15), label, fill=(255, 255, 255))

    frame_path = frame_dir / f'ndvi_{year}.png'
    img.save(frame_path)
    gif_frames.append(np.array(img))
    print(f'  {year}: frame saved')

gif_path = OUTPUT_DIR / 'bonus2_taroko_ndvi_26yr_timelapse.gif'
imageio.mimsave(gif_path, gif_frames, duration=0.65, loop=0)
print(f"  Saved GIF: {gif_path}")
display(IPImage(filename=str(gif_path)))

  2000: frame saved


  2001: frame saved


  2002: frame saved


  2003: frame saved


  2004: frame saved


...

  2024: frame saved


  2025: frame saved


  2026: frame saved


  Saved GIF: Week14_outputs\bonus2_taroko_ndvi_26yr_timelapse.gif


<IPython.core.display.Image object>

---
## S8b — Landsat × Sentinel-2 Cross-Sensor Analysis（跨感測器分析）✏️ EXERCISE

Landsat 看得遠（26 年），Sentinel-2 看得細（10m）。
現在你要結合兩者：用 Landsat 的長期趨勢找到變遷熱區，
再用 Sentinel-2 的高解析度驗證細節。

### 你需要完成的步驟：
1. 載入 Sentinel-2 資料（跟 W13 一樣的方法）
2. 計算 S2 的地震前後 ΔNDVI
3. 比較 Landsat vs S2 在重疊時段（2017–2026）的 NDVI 差異
4. 分析多解析度差異的原因

In [11]:
# ============================================================
# S8b — Cross-Sensor Analysis Exercise — COMPLETE
# ============================================================

# Step 1: Load Sentinel-2 (same cloud masking as W13)
def mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = scl.gte(4).And(scl.lte(7))
    return image.updateMask(mask).copyProperties(image, ['system:time_start'])

def add_s2_ndvi(image):
    # Sentinel-2: B8 = NIR, B4 = Red.
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI_S2')
    return image.addBands(ndvi).copyProperties(image, ['system:time_start'])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2017-04-01', DATE_END)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(mask_s2_clouds)
      .map(add_s2_ndvi))

s2_count = s2.size().getInfo()
landsat_overlap_count = landsat_idx.filterDate('2017-01-01', DATE_END).size().getInfo()
print(f'Sentinel-2 images loaded: {s2_count}')
print(f'Landsat images (same period): {landsat_overlap_count}')

Sentinel-2 images loaded: 247
Landsat images (same period): 381


In [12]:
# Step 2: Compute earthquake ΔNDVI from BOTH sensors

# Landsat pre/post composites and change.
l_pre  = landsat_idx.filterDate('2023-01-01', '2024-03-31').select('NDVI').median()
l_post = landsat_idx.filterDate('2024-04-01', '2024-12-31').select('NDVI').median()
l_change = l_post.subtract(l_pre).rename('delta_NDVI_landsat')

# Sentinel-2 pre/post composites and change.
s2_pre  = s2.filterDate('2023-01-01', '2024-03-31').select('NDVI_S2').median().rename('NDVI')
s2_post = s2.filterDate('2024-04-01', '2024-12-31').select('NDVI_S2').median().rename('NDVI')
s2_change = s2_post.subtract(s2_pre).rename('delta_NDVI_s2')

# Damage area comparison using the same threshold.
damage_threshold = -0.15
landsat_damage = l_change.lt(damage_threshold)
s2_damage = s2_change.lt(damage_threshold)

def damage_area_km2(mask, scale):
    return ee.Number(ee.Image.pixelArea().updateMask(mask).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=scale,
        maxPixels=1e10, tileScale=4
    ).get('area')).divide(1e6).getInfo()

cross_sensor_damage = pd.DataFrame([
    {'sensor': 'Landsat', 'resolution_m': 30, 'damage_area_km2': damage_area_km2(landsat_damage, 30)},
    {'sensor': 'Sentinel-2', 'resolution_m': 10, 'damage_area_km2': damage_area_km2(s2_damage, 10)},
])
cross_sensor_damage.to_csv(OUTPUT_DIR / 'bonus3_cross_sensor_damage_area.csv',
                           index=False, encoding='utf-8-sig')
display(cross_sensor_damage)

# Visualize both on the same map.
vis_delta = {'min': -0.3, 'max': 0.1,
             'palette': ['d73027', 'fc8d59', 'fee08b', 'ffffbf',
                         'd9ef8b', '91cf60', '1a9850']}

save_ee_thumbnail(
    l_change.clip(aoi),
    '06_landsat_delta_ndvi_2024.png',
    {'region': aoi, 'dimensions': 900, 'min': -0.3, 'max': 0.1, 'palette': vis_delta['palette']},
)
save_ee_thumbnail(
    s2_change.clip(aoi),
    '07_sentinel2_delta_ndvi_2024.png',
    {'region': aoi, 'dimensions': 900, 'min': -0.3, 'max': 0.1, 'palette': vis_delta['palette']},
)

Map_cs = geemap.Map(center=[24.18, 121.55], zoom=12)
Map_cs.addLayer(l_change.clip(aoi),  vis_delta, 'Landsat ΔNDVI (30m)')
Map_cs.addLayer(s2_change.clip(aoi), vis_delta, 'Sentinel-2 ΔNDVI (10m)')
Map_cs.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map_cs

,sensor,resolution_m,damage_area_km2
0,Landsat,30,70.067627
1,Sentinel-2,10,81.109061


  Saved thumbnail: Week14_outputs\06_landsat_delta_ndvi_2024.png


<IPython.core.display.Image object>

  Saved thumbnail: Week14_outputs\07_sentinel2_delta_ndvi_2024.png


<IPython.core.display.Image object>

Map(center=[24.18, 121.55], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [13]:
# Step 3: Annual NDVI cross-sensor comparison (2017–2026)

years_overlap = list(range(2017, 2027))
l_vals, s2_vals = [], []

for yr in years_overlap:
    # Landsat annual mean NDVI.
    l_yr = landsat_idx.filterDate(f'{yr}-01-01', f'{yr + 1}-01-01') \
        .select('NDVI').median()
    l_result = l_yr.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi,
        scale=100, maxPixels=1e9, tileScale=4
    ).getInfo()
    l_mean = l_result.get('NDVI')

    # Sentinel-2 annual mean NDVI.
    s2_yr_col = s2.filterDate(f'{yr}-01-01', f'{yr + 1}-01-01')
    s2_n = s2_yr_col.size().getInfo()
    if s2_n == 0:
        s_mean = None
    else:
        s_yr = s2_yr_col.select('NDVI_S2').median()
        s_result = s_yr.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi,
            scale=100, maxPixels=1e9, tileScale=4
        ).getInfo()
        s_mean = s_result.get('NDVI_S2')

    l_vals.append(l_mean)
    s2_vals.append(s_mean)

    l_str = f'{l_mean:.4f}' if l_mean is not None else 'N/A'
    s_str = f'{s_mean:.4f}' if s_mean is not None else 'N/A'
    print(f'  {yr}: Landsat={l_str}  S2={s_str}')

cross_sensor_df = pd.DataFrame({
    'year': years_overlap,
    'landsat_ndvi': l_vals,
    'sentinel2_ndvi': s2_vals,
}).dropna()
cross_sensor_df.to_csv(OUTPUT_DIR / 'bonus3_cross_sensor_annual_ndvi.csv',
                       index=False, encoding='utf-8-sig')

if len(cross_sensor_df) >= 2:
    r = np.corrcoef(cross_sensor_df['landsat_ndvi'], cross_sensor_df['sentinel2_ndvi'])[0, 1]
    r2 = r ** 2
else:
    r2 = np.nan

# --- Plot line comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years_overlap, l_vals, 's-', color='#8B4513',
        markersize=8, linewidth=2, label='Landsat (30m)')
ax.plot(years_overlap, s2_vals, 'o-', color='#2E8B57',
        markersize=8, linewidth=2, label='Sentinel-2 (10m)')
ax.axvline(2024, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Mean NDVI')
ax.set_title('Cross-Sensor NDVI: Landsat vs Sentinel-2 (2017–2026/03)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
line_fig = OUTPUT_DIR / 'bonus3_cross_sensor_ndvi_line.png'
plt.savefig(line_fig, dpi=180, bbox_inches='tight')
plt.show()

# --- Scatter with R² ---
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(cross_sensor_df['landsat_ndvi'], cross_sensor_df['sentinel2_ndvi'],
           s=70, color='#3B7A57', edgecolor='white')
for _, row in cross_sensor_df.iterrows():
    ax.annotate(str(int(row['year'])), (row['landsat_ndvi'], row['sentinel2_ndvi']),
                textcoords='offset points', xytext=(5, 4), fontsize=8)
lims = [
    min(cross_sensor_df['landsat_ndvi'].min(), cross_sensor_df['sentinel2_ndvi'].min()) - 0.02,
    max(cross_sensor_df['landsat_ndvi'].max(), cross_sensor_df['sentinel2_ndvi'].max()) + 0.02,
]
ax.plot(lims, lims, '--', color='gray', linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Landsat mean NDVI (30m)')
ax.set_ylabel('Sentinel-2 mean NDVI (10m)')
ax.set_title(f'Landsat vs Sentinel-2 Annual NDVI\n$R^2$ = {r2:.3f}')
ax.grid(True, alpha=0.3)
plt.tight_layout()
scatter_fig = OUTPUT_DIR / 'bonus3_landsat_s2_scatter.png'
plt.savefig(scatter_fig, dpi=180, bbox_inches='tight')
plt.show()

print(f"  Cross-sensor R²: {r2:.3f}")
print(f"  Saved figures: {line_fig}, {scatter_fig}")

  2017: Landsat=0.5421  S2=N/A


  2018: Landsat=0.5505  S2=0.4976


  2019: Landsat=0.5550  S2=0.5150


  2020: Landsat=0.5700  S2=0.5252


  2021: Landsat=0.5613  S2=0.5033


...

  2025: Landsat=0.5427  S2=0.5200


  2026: Landsat=0.4728  S2=0.5046


<Figure size 1000x500 with 1 Axes>

<Figure size 600x600 with 1 Axes>

  Cross-sensor R²: 0.309
  Saved figures: Week14_outputs\bonus3_cross_sensor_ndvi_line.png, Week14_outputs\bonus3_landsat_s2_scatter.png


### Bonus 3 Analysis

## Bonus 3: Landsat And Sentinel-2 Cross-Sensor Comparison

![Cross-sensor line](Week14_outputs/bonus3_cross_sensor_ndvi_line.png)

![Cross-sensor scatter](Week14_outputs/bonus3_landsat_s2_scatter.png)

跨感測器重疊時段使用 2017-2026/03 Sentinel-2 與 Landsat。Sentinel-2 共載入 **247 張**，同時段 Landsat 為 **381 張**。2018-2025 多數年份 Sentinel-2 年平均 NDVI 低於 Landsat；2026 因 Landsat 只有 1-3 月資料且影像數少，Landsat 0.4728 反而低於 Sentinel-2 0.5046。散佈圖 `R² = 0.309`，代表兩者有部分一致，但因解析度、雲遮罩、季節取樣與感測器光譜響應不同，不能直接把數值視為完全等價。

ΔNDVI < -0.15 的損害面積：

| Sensor | Resolution | Damage area (km²) |
|---|---:|---:|
| Landsat | 30 m | 70.07 |
| Sentinel-2 | 10 m | 81.11 |

Sentinel-2 面積較大，推測是 10 m 解析度能捕捉更多細碎崩塌面、道路切坡與河谷邊坡。Landsat 適合長期趨勢與大尺度熱區，Sentinel-2 適合震後細節定位；兩者合用才符合災害監測需求。

---
## S9 — Final Reflection（期末反思） ✅ COMPLETE

**Q1:** Compare the tools and time scales across W13 and W14:

| | W13 | W14 |
|---|---|---|
| Satellite | Sentinel-2 (10m) | Landsat (30m) |
| Time span | 6 years | 26 years |
| Best for | 震後細節、個別崩塌與道路/河谷小尺度變化 | 長期趨勢、世代尺度背景線、韌性與都市化變遷 |
| New technique | GEE 時序、median composite、Sentinel-1 SAR 時序 | 四代 Landsat band harmonization、QA_PIXEL bitmask、26 年 linearFit、MNDWI 水頻率、recovery ratio |

**Q2:** W6 used Kriging for **spatial** interpolation (points → surface).
W14 uses GEE for **temporal** analysis (snapshots → trends).
How do spatial and temporal perspectives complement each other in
disaster monitoring?

> **中文回答：** W6 的 Kriging 解決的是「空間缺口」：雨量站是點，但災害決策需要連續面，因此用空間插值推估站與站之間的降雨分布。W14 的 GEE 時序分析解決的是「時間缺口」：單張衛星影像只是快照，但災害與恢復是連續過程，因此用多年影像建立趨勢。兩者合在一起，就能同時回答「哪裡正在變」與「什麼時候開始變」；對防災而言，這比只看單點、單時刻更接近真實風險。

**Q3:** W14 uses two different study areas: Taroko (vegetation/resilience)
and Taoyuan (pond disappearance). Both demonstrate "long-term change
detection" but tell very different stories. What does each case reveal
about human-environment interaction over 26 years?

> **中文回答：** 太魯閣案例呈現的是自然擾動與地形控制下的生態韌性：地震、颱風、河流侵蝕與坡面不穩定會讓 NDVI 下降，但不同地形的恢復速度不同。桃園案例則是人為都市化改變水文景觀：埤塘被填平後，灌溉、防洪與文化景觀功能一起消失。兩者共同說明，長期遙測不只是看地表顏色變化，而是在追蹤人與環境如何互相塑造風險。

**Q4:** Looking back at W8–W14, if you could design the ideal monitoring
system for a mountain area like Taroko, which sensors, indices, and time
scales would you combine?

> **中文回答：** 理想系統會採三層架構：第一層用 Landsat 2000–至今建立 NDVI/NBR/MNDWI 長期趨勢與韌性背景線；第二層用 Sentinel-2 10 m 影像做震前震後 ΔNDVI、崩塌斑塊細部定位與復育追蹤；第三層用 Sentinel-1 SAR 補足雲雨季與夜間觀測，並用 VV/VH 或 coherence 交叉驗證裸露地與地表粗糙度變化。分析方法上，我會結合 W8 的光譜指標、W9 的變遷偵測、W10 的 SAR 穿雲能力、W12 的分類器，以及 W13/W14 的 GEE 時序分析，形成能同時看「狀態、變化、趨勢、韌性」的監測系統。

---

## Notebook Complete

```
S1: Setup → S2: Harmonize → S3: Indices → S4: Annual NDVI
→ S5: Trend Map → S6: Pond Disappearance → S7: Resilience
→ S8a: NDVI GIF → S8b: Cross-Sensor → S9: Reflection
```

> *"From snapshots to stories — 從快照到故事"*


---

# Appendix - Homework Report

# Week 14 Homework Report: ARIA v9.5 - The Resilience Monitor

**課程：** 遙測與空間資訊之分析與應用  
**主題：** Landsat 多年代趨勢分析、桃園埤塘消失與植被韌性監測  
**研究區：** 秀林/太魯閣山區；桃園台地  
**Notebook:** `Week14-Homework-Completed.ipynb` (homework submission); `Week14-Student.ipynb` (class exercise)  
**輸出資料夾：** `Week14_outputs/`

## 摘要

本週作業依照 0526 講義 ARIA v9.5 流程，將 W13 的 Sentinel-2 六年時序分析擴展為 Landsat L5/L7/L8/L9 的 2000-2026/03 長期監測。核心流程包含：四代 Landsat 波段調和、Collection 2 Level 2 反射率 scale factor 校正、`QA_PIXEL` bit 3/4 雲與雲影遮罩、年度 NDVI/MNDWI/NBR 指標、逐像素 `linearFit()` 趨勢、桃園 MNDWI 水頻率與埤塘消失偵測、2024 花蓮地震後 recovery ratio，以及 Landsat/Sentinel-2 跨感測器比較。

本次在太魯閣 AOI 共處理 **878 張 Landsat** 影像（L5 178、L7 387、L8 233、L9 80）。年度 NDVI 線性趨勢為 **+0.00188 NDVI/year**，2000-2026 趨勢估計增加 **+0.0488**，顯示整體為緩慢 greening；但逐像素分析仍有 **10.3%** 面積為 browning。桃園台地完整 AOI 估計 2000-2005 至 2021-2026/03 期間埤塘/水面淨減 **426.0 ha**；223 個已知埤塘點位中，MNDWI recent-water 偵測到 199 個，偵測率 **89.2%**。植被韌性分析顯示 2024 地震後顯著受損像元約 **91.94 km²**，平均 recovery ratio **0.415**，仍有 **57.1%** 受損像元屬退化或慢速恢復。

## Data And Workflow

- **Landsat collections：** `LANDSAT/LT05/C02/T1_L2`、`LANDSAT/LE07/C02/T1_L2`、`LANDSAT/LC08/C02/T1_L2`、`LANDSAT/LC09/C02/T1_L2`
- **時間範圍：** 2000-01-01 至 2026-03-31；2026 為部分年度，解讀時需特別標註。
- **Taroko AOI：** `[121.34526379253053, 24.046021742135874, 121.85149217685861, 24.35767637905926]`
- **Taoyuan full AOI：** `[120.94, 24.83, 121.35, 25.08]`
- **Taoyuan urban corridor：** `[121.00, 24.88, 121.28, 25.05]`
- **Band harmonization：** L5/L7 `SR_B1/B2/B3/B4/B5/B7` -> `Blue/Green/Red/NIR/SWIR1/SWIR2`；L8/L9 `SR_B2/B3/B4/B5/B6/B7` -> 同一組共同名稱。
- **Scale factor：** `DN * 0.0000275 + (-0.2)`，並 clamp 到 0-1 反射率範圍。
- **Cloud mask：** `QA_PIXEL` bit 3 為 cloud、bit 4 為 cloud shadow，兩者皆為 0 才保留。

## Task 1: Landsat Harmonization And 26-Year NDVI

![26-year NDVI](Week14_outputs/01_taroko_ndvi_26yr_timeseries.png)

Landsat 四代影像完成波段調和後，所有影像皆具有 `Blue, Green, Red, NIR, SWIR1, SWIR2, QA_PIXEL` 共同 band name，因此 NDVI 可用一致公式 `(NIR - Red) / (NIR + Red)` 計算。年度 NDVI 以每年 median composite 後取 AOI 空間平均。

主要結果：

| 指標 | 數值 |
|---|---:|
| Landsat total images | 878 |
| L5 / L7 / L8 / L9 | 178 / 387 / 233 / 80 |
| 26-year NDVI trend | +0.00188 NDVI/year |
| Trend-estimated total change | +0.0488 |
| Highest annual mean NDVI | 2022, 0.5886 |
| Lowest annual mean NDVI | 2026, 0.4728 |
| 2024 earthquake year mean NDVI | 0.5607 |

2000-2026/03 的整體趨勢是緩慢 greening。2005、2007、2012 與 2026 有較低 NDVI，其中 2026 只到 3 月，影像數僅 8 張，受季節與資料不完整影響，不應直接視為全年退化。2009 莫拉克年平均 NDVI 為 0.5152，2024 地震年為 0.5607；兩者在 AOI 年平均折線上並非最強低谷，說明災害訊號被大面積森林與季節差異稀釋。因此，事件影響應搭配逐像素趨勢、ΔNDVI 與 recovery ratio 判讀，而不是只看整體平均。

## Task 2: Pixel-Level Linear Trend Analysis

![NDVI slope map](Week14_outputs/02_taroko_ndvi_slope_map.png)

本段依講義使用年度 NDVI ImageCollection 加入 `time` band，再以 `ee.Reducer.linearFit()` 對每個像素計算 2000-2026/03 的 NDVI slope。分類門檻設為：

- `slope > 0.001`：greening
- `-0.001 <= slope <= 0.001`：stable
- `slope < -0.001`：browning

結果如下：

| Class | Area (km²) | Percent |
|---|---:|---:|
| Greening | 1351.34 | 76.1% |
| Browning | 183.11 | 10.3% |
| Stable | 240.84 | 13.6% |
| Valid total | 1775.29 | 100.0% |

這個結果與 Task 1 的整體 greening 一致，但空間圖顯示 browning 不是不存在，而是集中在特定位置，例如河谷、道路切坡、裸露坡面與持續受侵蝕區。W13 的 Sentinel-2 六年趨勢適合回答「地震後哪裡變化最明顯」，但 W14 的 Landsat 二十六年趨勢能判斷「這個像元是長期退化，還是短期事件造成」。因此，W13 與 W14 的差異不是誰取代誰，而是短期細節與長期背景的互補。

GeoTIFF Drive export 已啟動並完成：

- `Week14_taroko_ndvi_trend_26yr` - `SUCCEEDED`

## Task 3: Taoyuan Pond Disappearance With MNDWI

![Taoyuan water frequency](Week14_outputs/03_taoyuan_water_frequency.png)

![Taoyuan pond change](Week14_outputs/04_taoyuan_pond_change_map.png)

桃園分析使用 MNDWI `(Green - SWIR1) / (Green + SWIR1)`，並以 `MNDWI > 0.1` 判為水體。先建立 2000-2026/03 每年水體 mask，再計算水頻率圖；接著比較 early period（2000-2005）與 recent period（2021-2026/03）之 median MNDWI water mask。

面積結果：

| Region | Stable water (ha) | Lost ponds (ha) | New water (ha) | Net change (ha) |
|---|---:|---:|---:|---:|
| Taoyuan full | 19047.56 | 588.80 | 162.80 | -425.99 |
| Urban corridor | 4034.77 | 352.19 | 50.63 | -301.57 |

完整桃園 AOI 的淨水面變化為 **-426.0 ha**，其中中壢/新屋/高鐵走廊就佔 **-301.6 ha**，顯示消失埤塘高度集中在都市化走廊。新增水體面積小於消失面積，可能包含新設滯洪池、工程開挖積水、整修後農塘或季節性水田；因此不宜直接把所有 green 像元解讀成永久水塘增加。

驗證結果：

| Known ponds | MNDWI detected | Detection rate |
|---:|---:|---:|
| 223 | 199 | 89.2% |

MNDWI 對大部分已知埤塘有效，但可能漏掉小面積水體、混合像元、植被覆蓋水面或陰影干擾。都市防洪上，埤塘被填平代表分散式蓄洪容量下降，不透水面增加，使短延時強降雨下的逕流尖峰更高，排水系統更容易超載。這也是講義強調桃園埤塘不只是文化景觀，也是都市韌性基礎設施的原因。

## Task 4: Vegetation Resilience Metrics

![Recovery ratio](Week14_outputs/05_taroko_recovery_ratio_map.png)

依講義定義三期：

- **Baseline：** 2020-01-01 至 2024-03-31
- **Impact：** 2024-04-01 至 2024-12-31
- **Recovery：** 2025-06-01 至 2026-03-31

公式：

```text
Recovery Ratio = (Recovery_NDVI - Impact_NDVI) / (Baseline_NDVI - Impact_NDVI)
```

本作業只評估 `Baseline - Impact > 0.1` 的顯著受損像元。結果如下：

| Class | Area (km²) | Percent |
|---|---:|---:|
| Degrading (< 0) | 16.15 | 17.6% |
| Slow recovery (0-0.5) | 36.36 | 39.5% |
| Recovering (0.5-1.0) | 27.17 | 29.6% |
| Exceeded (> 1.0) | 11.93 | 13.0% |
| Damaged pixels evaluated | 91.94 | 100.0% |
| Mean recovery ratio | 0.415 | |

受損像元中，退化與慢速恢復合計 **57.1%**，代表超過一半的顯著受損地區尚未恢復到震前一半以上。這些區域多半是陡峭峽谷壁、河道兩側裸露堆積、道路切坡、崩塌源頭或持續受沖刷的坡面；即使經過 2025-2026 的恢復期，NDVI 仍偏低。相對地，recovering 與 exceeded 區域可能位於較緩坡面、谷地邊緣或仍保有土壤的崩塌堆積面，植被可快速長草或灌木化。

復育建議是分級處理：靠近道路、溪流、聚落或保全對象，且 recovery ratio 低的地區應優先工程穩定與主動復育；recovery ratio 已高且遠離保全對象的區域，則可採自然恢復監測，避免過度介入。

GeoTIFF Drive export 已啟動並完成：

- `Week14_taroko_recovery_ratio` - `SUCCEEDED`

## Bonus 1: Multi-Index Dashboard

![Multi-index dashboard](Week14_outputs/bonus1_multi_index_dashboard.png)

本次同時計算 NDVI、MNDWI、NBR。NDVI 長期緩慢上升，但 NBR 在 2014 以後波動較明顯，2024-2026 更低，顯示 NBR 對裸露、擾動或 SWIR2 反應更敏感。MNDWI 在太魯閣 AOI 多為負值，符合山區植被/裸地為主、水面比例較低的特性；若要分析水體變化，桃園埤塘案例比太魯閣整體 AOI 更適合。

## Bonus 2: NDVI 26-Year Time-Lapse Animation

產出 GIF：

- `Week14_outputs/bonus2_taroko_ndvi_26yr_timelapse.gif`

動畫共 27 幀（2000-2026），每幀標示年份，並特別標記 2009 Morakot 與 2024 Hualien EQ。最明顯的三個視覺重點是：第一，太魯閣山區長期仍以高 NDVI 森林為主；第二，河谷、海岸與裸露坡面在多年中反覆呈現低 NDVI；第三，2024 後局部紅黃區變多，但整體山區並非全面崩壞，需靠 recovery ratio 才能判斷哪些受損地真正未恢復。

## Bonus 3: Landsat And Sentinel-2 Cross-Sensor Comparison

![Cross-sensor line](Week14_outputs/bonus3_cross_sensor_ndvi_line.png)

![Cross-sensor scatter](Week14_outputs/bonus3_landsat_s2_scatter.png)

跨感測器重疊時段使用 2017-2026/03 Sentinel-2 與 Landsat。Sentinel-2 共載入 **247 張**，同時段 Landsat 為 **381 張**。2018-2025 多數年份 Sentinel-2 年平均 NDVI 低於 Landsat；2026 因 Landsat 只有 1-3 月資料且影像數少，Landsat 0.4728 反而低於 Sentinel-2 0.5046。散佈圖 `R² = 0.309`，代表兩者有部分一致，但因解析度、雲遮罩、季節取樣與感測器光譜響應不同，不能直接把數值視為完全等價。

ΔNDVI < -0.15 的損害面積：

| Sensor | Resolution | Damage area (km²) |
|---|---:|---:|
| Landsat | 30 m | 70.07 |
| Sentinel-2 | 10 m | 81.11 |

Sentinel-2 面積較大，推測是 10 m 解析度能捕捉更多細碎崩塌面、道路切坡與河谷邊坡。Landsat 適合長期趨勢與大尺度熱區，Sentinel-2 適合震後細節定位；兩者合用才符合災害監測需求。

## Cross-Week Integration Summary

W6 的 Kriging 解決空間缺口：雨量站是點，但防災需要連續雨量面。W14 的 GEE Landsat 時序解決時間缺口：單張影像是快照，但地震、颱風、都市化與復育都是跨年的過程。兩者合在一起，能建立 `x, y, time, value` 的時空觀點。

從 W8 到 W14，ARIA 的思維由「單張影像裡有什麼」逐步變成「這個地景如何演變」。W8 單景 NDVI 能判斷植被狀態，W9 兩景 ΔNDVI 能看變化，W10 SAR 補足雲雨與夜間觀測，W12 分類器能把地表狀態轉為土地覆蓋類別，W13 GEE Sentinel-2 建立六年高解析時序，W14 Landsat 則補上二十六年背景線與韌性。短期 Sentinel-2 看到的是地震後細節，長期 Landsat 則回答「這些變化是否超出歷史脈絡」。

限制也很明確：Landsat 30 m 會混合小型崩塌、道路、水體與陰影，不如 Sentinel-2 10 m 精細；L7 2003 後 SLC-off 可能造成條帶缺口，雖然 annual median 可減輕但不能完全消除；山區雲量高，使部分年份有效觀測較少，2026 又是部分年度。因此，本作業的結論應視為雲端遙測的長期監測成果，後續若要支援工程決策，仍需高解析影像、DEM slope/aspect、道路/溪流資料與現地調查交叉驗證。

## Output Checklist

- `Week14-Student.ipynb` - 已完成並執行，13 個 code cells 皆有輸出，0 個錯誤輸出。
- `Week14_outputs/00_week14_contact_sheet.png`
- `Week14_outputs/01_taroko_ndvi_26yr_timeseries.png`
- `Week14_outputs/02_taroko_ndvi_slope_map.png`
- `Week14_outputs/03_taoyuan_water_frequency.png`
- `Week14_outputs/04_taoyuan_pond_change_map.png`
- `Week14_outputs/05_taroko_recovery_ratio_map.png`
- `Week14_outputs/06_landsat_delta_ndvi_2024.png`
- `Week14_outputs/07_sentinel2_delta_ndvi_2024.png`
- `Week14_outputs/bonus1_multi_index_dashboard.png`
- `Week14_outputs/bonus2_taroko_ndvi_26yr_timelapse.gif`
- `Week14_outputs/bonus3_cross_sensor_ndvi_line.png`
- `Week14_outputs/bonus3_landsat_s2_scatter.png`
- `Week14_outputs/task1_annual_ndvi_stats.csv`
- `Week14_outputs/task2_trend_area_stats.csv`
- `Week14_outputs/task3_pond_area_stats.csv`
- `Week14_outputs/task3_pond_detection_rate.csv`
- `Week14_outputs/task4_recovery_ratio_stats.csv`

## Validation Status

Final check: **PASS**.

- Notebook executed in place with sequential execution counts 1-13.
- No notebook error outputs.
- No source placeholders or answer-template remnants remain in the notebook source.
- Key output images visually checked via contact sheet.
- GEE export tasks `Week14_taroko_ndvi_trend_26yr` and `Week14_taroko_recovery_ratio` both reached `SUCCEEDED`.
